# Paraphrasing Judge

In [36]:
import os
import json
import time
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

client = OpenAI(api_key=API_KEY)

INPUT_PATH = "sampled_from_defan_1000_NOMATH_8B_temp0.8_topp0.95_maxtokens32.jsonl"
OUTPUT_PATH = "judged_labels_checkpoint.jsonl"

MODEL = "gpt-4.1-mini"
SAVE_EVERY = 20
MAX_RETRIES = 5

df = pd.read_json(INPUT_PATH, lines=True)

SYSTEM = """You are an expert evaluator, judging correctness of a given LLM-generated responses to given questions.

You will be given:
Q: question
R: reference answer (may list multiple required items)
A: model answer

Decide whether A is correct.

Return ONLY valid JSON:
{"label": 0} or {"label": 1}

Label 0 (correct) if:
- A directly answers Q.
- A matches R in semantic meaning. Synonyms/aliases/paraphrases/different languages etc. are acceptable; order and phrasing do not matter.
- If an answer is expressed in slightly more detail/with slightly more descriptors than expected, that is okay as long as it is factually accurate.
- If R contains multiple required items/entities, A includes ALL of them.

Numeric tolerance:
- For numeric answers, reasonable approximations are acceptable.
- Dates and years however must be exact.

Label 1 (incorrect) if ANY:
- A is empty, a refusal, or not an answer.
- A is missing any required item from R.
- A contradicts Q or R (wrong entity/year/place/value), even if it also mentions a correct token.
- A merely mentions the correct answer string without asserting it as the answer."""

def append_jsonl(rows, path):
    with open(path, "a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

def judge_one(q, r, a):
    a = "" if a is None else str(a).strip()
    if not a:
        return 1

    prompt = f'Q:{q}\nR:{r}\nA:{a}\nReturn JSON:{{"label":0|1}}'

    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=0,
                max_completion_tokens=30,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": prompt},
                ],
            )
            obj = json.loads(resp.choices[0].message.content)
            return 0 if obj.get("label", 1) == 0 else 1

        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print(f"Failed on one row after retries: {e}")
                return 1
            time.sleep(2 ** attempt)

done_ids = set()
if os.path.exists(OUTPUT_PATH):
    old = pd.read_json(OUTPUT_PATH, lines=True)
    done_ids = set(old["sample_id"].tolist())
    print(f"Found checkpoint with {len(done_ids)} completed rows")

remaining_df = df[~df["sample_id"].isin(done_ids)].copy()
print(f"Rows remaining: {len(remaining_df)}")

rows_to_save = []

for _, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="Judging"):
    label = judge_one(row["question"], row["answer"], row["model_answer"])

    rows_to_save.append({
        "row_id": row["sample_id"],
        "label": label
    })

    if len(rows_to_save) >= SAVE_EVERY:
        append_jsonl(rows_to_save, OUTPUT_PATH)
        rows_to_save = []

if rows_to_save:
    append_jsonl(rows_to_save, OUTPUT_PATH)

labels_df = pd.read_json(OUTPUT_PATH, lines=True)
labels_df

Rows remaining: 15000


Judging:   0%|          | 0/15000 [00:00<?, ?it/s]

,row_id,label
0,C0001_q2_s01,1
1,C0001_q2_s02,1
2,C0001_q2_s03,1
3,C0001_q2_s04,0
4,C0001_q2_s05,1
5,C0001_q2_s06,0
6,C0001_q2_s07,1
7,C0001_q2_s08,1
8,C0001_q2_s09,0
9,C0001_q2_s10,1


In [37]:
labels_df["label"].value_counts()

label
1    9058
0    5942
Name: count, dtype: int64

In [38]:
labels_df.to_csv("sample_labels.csv",index=False)